# Phase 3: Pareto-Dominance DPO Training (Kaggle 2xT4)

Trains a DPO LoRA adapter on `experiments/014_pareto_dpo/data/pareto_dpo_pairs.jsonl` (154 pairs, built locally by `scripts/data_prep/format_pareto_dpo.py` from experiment 011's full-scale dynamic multi-criteria judge scores -- no new API calls needed). Unlike the existing Full DPO run (`experiments/001_500_reasoning/data/dpo_pairs.jsonl`, single-scalar all-steps-pass selection), each pair here is only kept when the correct rollout Pareto-dominates the incorrect one across all 9 reward-tree categories at once -- at least as good on every category, strictly better on one -- so the preference signal excludes cases where the two rollouts trade off against each other on different failure axes.

Same trainer (`scripts/train/train_dpo.py`), same base model, same hyperparameters as the existing Full DPO kernel -- the only variable being tested is pair-selection method.


In [ ]:
# Cell 1: Check GPU hardware and install sm_60 compatible PyTorch stack if Tesla P100 is assigned
# (identical to the proven-working kaggle_train_dpo/train_dpo.ipynb cell 1 -- no changes)
import os, subprocess, sys, torch

print(f'Initial PyTorch: {torch.__version__}')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    cc = torch.cuda.get_device_capability(0)
    print(f'GPU: {props.name}, Compute Capability: {cc}, VRAM: {props.total_memory / 1e9:.1f} GB')
    if cc[0] < 7:
        print(f'*** Tesla P100 (cc {cc}) detected. PyTorch 2.12 dropped sm_60 CUDA kernels.')
        print('*** Installing PyTorch 2.5.1+cu124 with full sm_60 CUDA GPU support...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.5.1', 'torchvision==0.20.1', '--index-url', 'https://download.pytorch.org/whl/cu124'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=False)
packages = ['transformers==4.49.0', 'peft==0.14.0', 'accelerate==1.2.1', 'qwen-vl-utils==0.0.14', 'pillow']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *packages], check=True)

import torch, transformers
print(f'Active PyTorch: {torch.__version__}, CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'Active GPU device: {torch.cuda.get_device_name(0)}')
print('Packages successfully configured.')


In [ ]:
# Cell 2: Checkout repository and execute Pareto-dominance DPO trainer
import os, subprocess, sys
from pathlib import Path

repo_dir = Path('/tmp/chart-prm')
if repo_dir.exists():
    subprocess.run(['rm', '-rf', str(repo_dir)], check=True)

subprocess.run(['git', 'clone', 'https://github.com/yahorlahunovich/chart-prm.git', str(repo_dir)], check=True)
# chart_prm.pareto and experiments/014_pareto_dpo/data/pareto_dpo_pairs.jsonl (this session's
# Phase 3 work) only exist on the ertugrul branch, not yet merged to main -- a plain clone
# defaults to main, which would fail here exactly like it did on a different kernel earlier
# this session (chartgemma-holdout: ModuleNotFoundError: chart_prm.text_match on main).
subprocess.run(['git', '-C', str(repo_dir), 'checkout', 'ertugrul'], check=True)
os.chdir(repo_dir)
print(f'Working directory set to {repo_dir}')

pairs_path = repo_dir / 'experiments/014_pareto_dpo/data/pareto_dpo_pairs.jsonl'
assert pairs_path.exists(), f'Missing {pairs_path} after checkout -- branch or commit problem.'
with open(pairs_path, encoding='utf-8') as f:
    n = sum(1 for line in f if line.strip())
assert n == 154, f'Expected 154 pairs, found {n}'
print(f'Confirmed {n} Pareto DPO pairs present after checkout.')

# Run Pareto-dominance DPO training -- same trainer, same hyperparameters as the existing
# Full DPO kernel, only --dataset-path/--output-dir differ.
env = os.environ.copy()
env['PYTHONPATH'] = 'src'
cmd = [
    sys.executable, 'scripts/train/train_dpo.py',
    '--dataset-path', 'experiments/014_pareto_dpo/data/pareto_dpo_pairs.jsonl',
    '--output-dir', '/kaggle/working/qwen_vl_pareto_dpo_adapter',
    '--epochs', '1',
    '--batch-size', '1',
    '--lr', '1e-5',
    '--beta', '0.1'
]
subprocess.run(cmd, env=env, check=True)


In [ ]:
# Cell 3: Validate output artifacts
out_dir = Path('/kaggle/working/qwen_vl_pareto_dpo_adapter')
files = sorted([p.name for p in out_dir.iterdir()]) if out_dir.exists() else []
print(f'Adapter directory {out_dir} contents: {files}')
assert (out_dir / 'adapter_config.json').exists(), 'adapter_config.json missing -- training did not save a LoRA adapter.'
